# Training VQGAN

In [10]:
import sys
import zarr
import io
import os
import numpy as np
import torch
import zipfile
import bisect
import pytorch_lightning as pl
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split
from pytorch_lightning.callbacks import ModelCheckpoint
from omegaconf import OmegaConf
from zarr.storage import ZipStore
from torchvision.utils import save_image


class FolderZarrDataset(Dataset):
    def __init__(self, folder_path, max_per_day=None, downsample=10):
        """
        folder_path: Path to folder containing .zarr.zip files
        max_per_day: optional limit on number of images per file
        """
        folder_path = Path(folder_path)
        zarr_files = sorted(folder_path.glob("*.zip"))  # all .zip files
        if not zarr_files:
            raise ValueError(f"No .zip files found in {folder_path}")

        self.arrays = []
        self.lengths = []

        for path in zarr_files:
            # automatically compute nested dataset path
            zarr_name = path.name.replace(".zip", "")
            dataset_name = f"{zarr_name}/images"

            store = ZipStore(str(path), mode="r")
            arr = zarr.open(store=store, path=dataset_name, mode="r")

            print(f"Loaded {path} with shape {arr.shape}")

            if max_per_day is not None:
                arr = arr[:max_per_day]

            arr = arr[::downsample]

            self.arrays.append(arr)
            self.lengths.append(arr.shape[0])

        # cumulative lengths for indexing
        self.cum_lengths = []
        total = 0
        for l in self.lengths:
            total += l
            self.cum_lengths.append(total)

    def __len__(self):
        return self.cum_lengths[-1]

    def __getitem__(self, idx):
        # find which day
        day_idx = bisect.bisect_right(self.cum_lengths, idx)
        local_idx = idx if day_idx == 0 else idx - self.cum_lengths[day_idx - 1]

        img = self.arrays[day_idx][local_idx]  # (C,H,W)
        img = torch.from_numpy(img).float()
        img = img / 127.5 - 1.0  # normalize [-1,1]

        return img


class VQForOrders(pl.LightningModule):
    """VQGAN model class, adapted for Orders"""

    def __init__(self, vqmodel, lr=1e-4):
        super().__init__()
        self.m = vqmodel
        self.lr = lr

    def forward(self, x):
        q, _, info = self.m.encode(x)
        return self.m.decode(q)

    def training_step(self, batch, _):
        x = batch
        x_rec = self(x)
        loss = ((x_rec - x) ** 2).mean()
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x = batch
        x_rec = self(x)
        val_loss = ((x_rec - x) ** 2).mean()
        self.log("val_loss", val_loss, prog_bar=True)
        return val_loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)


# ---------- Train ----------
def train():

    THIS_DIR = Path(os.getcwd()).resolve().parent  # custom_code/training
    REPO = THIS_DIR.parents[1]  # repo root (MarS-main)
    print(REPO.name)

    # access the VQGAN official code
    sys.path.insert(0, str(REPO / "Mars_Derrick" / "third_party" / "latent-diffusion"))
    sys.path.insert(0, str(REPO / "Mars_Derrick" / "third_party" / "taming-transformers"))

    from ldm.util import instantiate_from_config

    # Build the new path in /scratch
    SCRATCH_DATASET_PATH = Path("/scratch") / REPO.name / "Data_zarr" / "Training_VQGAN"

    print("Using dataset path:", SCRATCH_DATASET_PATH)

    MODEL_CKPT_PATH = Path("/scratch") / REPO.name

    print("Using dataset path:", MODEL_CKPT_PATH)

    # Update your dataset initialization

    dataset = FolderZarrDataset(SCRATCH_DATASET_PATH, downsample=1000)

    ref = dataset[0]

    all_identical = True
    for i in range(1, len(dataset)):
        if not torch.equal(ref, dataset[i]):
            all_identical = False
            print(f"Mismatch at index {i}")
            break

    ref = dataset[0]
    x = dataset[12]

    diff = x - ref

    print("Max abs diff:", diff.abs().max())
    print("Mean abs diff:", diff.abs().mean())
    print("Are equal:", torch.equal(ref, x))
    print("Are close:", torch.allclose(ref, x, atol=1e-6))

    print("All samples identical:", all_identical)

    # `dataset` is your FolderZarrDataset or MultiZarrImageDataset
    n = len(dataset)

    # Compute sizes
    n_train = int(0.8 * n)
    n_val = n - n_train  # remainder goes to validation

    # Split dataset
    train_ds, val_ds = random_split(
        dataset,
        [n_train, n_val],
        generator=torch.Generator().manual_seed(0),  # reproducible split
    )

    # Create DataLoaders
    train_dl = DataLoader(
        train_ds,
        batch_size=64,
        shuffle=True,  # shuffle training data
        num_workers=4,
        drop_last=True,  # drop last batch if smaller than batch_size,
        persistent_workers=True,
    )

    val_dl = DataLoader(
        val_ds,
        batch_size=64,
        shuffle=False,  # no shuffle for validation
        num_workers=4,
        persistent_workers=True,
    )

    print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

    # load VQGan model with config and weights as in MarS paper
    cfg = OmegaConf.load(str(REPO / "Mars_Derrick/third_party/latent-diffusion/models/first_stage_models/vq-f4/config.yaml"))

    vq = instantiate_from_config(cfg.model)

    if torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("Using device:", device)

    # Loading pretrained model checkpoint
    ZIP_PATH = MODEL_CKPT_PATH / "vq-f4.zip"

    # Load model.ckpt directly from zip without extracting
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        # Read the ckpt file into memory as bytes
        with zip_ref.open("model.ckpt") as ckpt_file:
            ckpt_bytes = ckpt_file.read()  # load all bytes into memory
            ckpt_buffer = io.BytesIO(ckpt_bytes)  # create a buffer

    # Load the checkpoint with torch directly from buffer
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("Model checkpoint loaded successfully from zip (without extracting)!")
    vq.load_state_dict(ckpt.get("state_dict", ckpt), strict=False)
    vq.learning_rate = 1e-4
    vq.train()

    model = VQForOrders(vq, lr=1e-4)

    ckpt_cb = ModelCheckpoint(
        dirpath=THIS_DIR / "checkpoints",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
        filename="best-{epoch:03d}-{val_loss:.6f}",
    )

    trainer = pl.Trainer(
        default_root_dir=str(THIS_DIR),
        accelerator="gpu" if torch.cuda.is_available() or torch.backends.mps.is_available() else "cpu",
        devices=1,
        max_epochs=20,
        check_val_every_n_epoch=2,  # <-- val every X epochs
        log_every_n_steps=50,
        callbacks=[ckpt_cb],
    )

    trainer.fit(model, train_dl, val_dl)
    print("Best checkpoint:", ckpt_cb.best_model_path)

    """

    # Post training, test one sample using BEST weights + save gt/pred
    best = VQForOrders.load_from_checkpoint(ckpt_cb.best_model_path, vqmodel=vq, lr=1e-4)
    best.eval()
    best.to("cuda" if torch.cuda.is_available() else "cpu")

    x = test_ds[0].unsqueeze(0).to(best.device)  # (1,3,32,32)
    with torch.no_grad():
        x_rec = best(x)

    # Save as images (map [-1,1] -> [0,1])
    gt = (x[0].cpu() + 1) / 2
    pr = (x_rec[0].cpu() + 1) / 2
    save_image(gt, OUT_DIR / "gt.png")
    save_image(pr, OUT_DIR / "pred.png")
    print("Saved:", OUT_DIR / "gt.png", "and", OUT_DIR / "pred.png")
    """


if __name__ == "__main__":
    train()

project_2012747
Using dataset path: /scratch/project_2012747/Data_zarr/Training_VQGAN
Using dataset path: /scratch/project_2012747
Loaded /scratch/project_2012747/Data_zarr/Training_VQGAN/LOBSTER-AAPL-2025-12-17_order_images.zarr.zip with shape (5227808, 3, 32, 32)
Loaded /scratch/project_2012747/Data_zarr/Training_VQGAN/LOBSTER-AMZN-2025-12-09_order_images.zarr.zip with shape (2637741, 3, 32, 32)
Loaded /scratch/project_2012747/Data_zarr/Training_VQGAN/LOBSTER-AMZN-2025-12-11_order_images.zarr.zip with shape (3812235, 3, 32, 32)
Loaded /scratch/project_2012747/Data_zarr/Training_VQGAN/LOBSTER-GOOGL-2025-12-16_order_images.zarr.zip with shape (18751982, 3, 32, 32)
Loaded /scratch/project_2012747/Data_zarr/Training_VQGAN/LOBSTER-TSLA-2025-12-12_order_images.zarr.zip with shape (10442283, 3, 32, 32)
Loaded /scratch/project_2012747/Data_zarr/Training_VQGAN/LOBSTER-TSLA-2025-12-17_order_images.zarr.zip with shape (10859624, 3, 32, 32)
Loaded /scratch/project_2012747/Data_zarr/Training_VQGA

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Model checkpoint loaded successfully from zip (without extracting)!


┏━━━┳━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name ┃ Type    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ m    │ VQModel │ 72.8 M │ train │     0 │
└───┴──────┴─────────┴────────┴───────┴───────┘

Trainable params: 58.1 M                                                                                           
Non-trainable params: 14.7 M                                                                                       
Total params: 72.8 M                                                                                               
Total estimated model params size (MB): 291                                                                        
Modules in train mode: 246                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

# Testing the Best model

In [54]:
import random
import torch
import math
from pathlib import Path
import re
import pytorch_lightning as pl
from pathlib import Path
import os
from omegaconf import OmegaConf
from zarr.storage import ZipStore
import zarr
import bisect
from torchvision.utils import save_image
from torchvision.utils import save_image
from torch.utils.data import Dataset, DataLoader, random_split


class FolderZarrDataset(Dataset):
    def __init__(self, folder_path, max_per_day=None, downsample=10):
        """
        folder_path: Path to folder containing .zarr.zip files
        max_per_day: optional limit on number of images per file
        """
        folder_path = Path(folder_path)
        zarr_files = sorted(folder_path.glob("*.zip"))  # all .zip files
        if not zarr_files:
            raise ValueError(f"No .zip files found in {folder_path}")

        self.arrays = []
        self.lengths = []

        for path in zarr_files:
            # automatically compute nested dataset path
            zarr_name = path.name.replace(".zip", "")
            dataset_name = f"{zarr_name}/images"

            store = ZipStore(str(path), mode="r")
            arr = zarr.open(store=store, path=dataset_name, mode="r")

            print(f"Loaded {path} with shape {arr.shape}")

            if max_per_day is not None:
                arr = arr[:max_per_day]

            arr = arr[::downsample]

            self.arrays.append(arr)
            self.lengths.append(arr.shape[0])

        # cumulative lengths for indexing
        self.cum_lengths = []
        total = 0
        for l in self.lengths:
            total += l
            self.cum_lengths.append(total)

    def __len__(self):
        return self.cum_lengths[-1]

    def __getitem__(self, idx):
        # find which day
        day_idx = bisect.bisect_right(self.cum_lengths, idx)
        local_idx = idx if day_idx == 0 else idx - self.cum_lengths[day_idx - 1]

        img = self.arrays[day_idx][local_idx]  # (C,H,W)
        img = torch.from_numpy(img).float()
        img = img / 127.5 - 1.0  # normalize [-1,1]

        return img


def load_best_model(best_ckpt_path, repo_root):
    from ldm.util import instantiate_from_config

    cfg = OmegaConf.load(repo_root / "Mars_Derrick/third_party/latent-diffusion/models/first_stage_models/vq-f4/config.yaml")

    vq = instantiate_from_config(cfg.model)

    model = VQForOrders.load_from_checkpoint(checkpoint_path=best_ckpt_path, vqmodel=vq, lr=1e-4, strict=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

    model.to(device)
    model.eval()

    return model, device


def find_best_checkpoint(ckpt_dir: Path) -> Path:
    ckpts = list(ckpt_dir.glob("*.ckpt"))
    if not ckpts:
        raise FileNotFoundError(f"No .ckpt files found in {ckpt_dir}")

    def extract_val_loss(p):
        m = re.search(r"val_loss=([0-9]+(?:\.[0-9]+)?)", p.name)
        if m is None:
            raise ValueError(f"Could not parse val_loss from filename: {p.name}")
        return float(m.group(1))

    best_ckpt = min(ckpts, key=extract_val_loss)
    return best_ckpt


def test_single_sample():
    THIS_DIR = Path(os.getcwd()).resolve().parent
    REPO = THIS_DIR.parents[1]
    setup_third_party_paths(REPO)

    BEST_CKPT = Path("/scratch") / REPO.name / "Mars_Derrick" / "checkpoints" / "checkpoint_downsample_100"
    DATA_Path = Path("/scratch") / REPO.name / "Data_zarr" / "Testing_VQGAN"
    OUT_DIR = THIS_DIR / "inference"
    OUT_DIR.mkdir(exist_ok=True)

    # Load model
    model, device = load_best_model(BEST_CKPT, REPO)

    # Load ONE sample only
    print(DATA_Path)
    sample_ds = FolderZarrDataset(DATA_Path, downsample=100000)
    x = sample_ds[0].unsqueeze(0).to(device)

    with torch.no_grad():
        x_rec = model(x)

    print("Input sample", type(x), x.shape, x)

    # Save images
    gt = (x[0].cpu() + 1) / 2
    pr = (x_rec[0].cpu() + 1) / 2

    l1 = torch.mean(torch.abs(pr - gt)).item()
    print("L1 error:", l1)

    save_image(gt, OUT_DIR / "gt.png")
    save_image(pr, OUT_DIR / "pred.png")

    print("Saved inference results to:", OUT_DIR)


def test_random_samples(n=100, save_first=True):
    THIS_DIR = Path(os.getcwd()).resolve().parent
    REPO = THIS_DIR.parents[1]
    setup_third_party_paths(REPO)

    BEST_CKPT_PATH = Path("/scratch") / REPO.name / "Mars_Derrick" / "checkpoints" / "checkpoint_downsample_100/"

    BEST_CKPT = find_best_checkpoint(BEST_CKPT_PATH)

    DATA_PATH = Path("/scratch") / REPO.name / "Data_zarr" / "Testing_VQGAN"
    OUT_DIR = THIS_DIR / "inference"
    OUT_DIR.mkdir(exist_ok=True)

    # ---- Load model ----
    model, device = load_best_model(BEST_CKPT, REPO)
    model.eval()

    # ---- Dataset (same as training, but no DataLoader) ----
    dataset = FolderZarrDataset(DATA_PATH, downsample=1000)

    # ---- Random indices ----
    n = min(n, len(dataset))
    indices = random.sample(range(len(dataset)), n)

    l1_list = []
    mse_list = []
    psnr_list = []

    with torch.no_grad():
        for i, idx in enumerate(indices):
            x = dataset[idx].unsqueeze(0).to(device)
            x_rec = model(x)

            # saving just to analysis
            if i == 0:
                x_saved = x.clone()  # clone + move to CPU
                x_rec_saved = x_rec.clone()

            # map to [0,1]
            gt = (x[0] + 1) / 2
            pr = (x_rec[0] + 1) / 2

            l1 = torch.mean(torch.abs(pr - gt)).item()
            mse = torch.mean((pr - gt) ** 2).item()
            psnr = 20 * math.log10(1.0 / math.sqrt(mse))  # peak signal to noise ratio

            l1_list.append(l1)
            mse_list.append(mse)
            psnr_list.append(psnr)

            # Optionally save the first sample for visual sanity check
            if save_first and i == 0:
                save_image(gt.cpu(), OUT_DIR / "gt.png")
                save_image(pr.cpu(), OUT_DIR / "pred.png")

    # ---- Final averaged metrics ----

    avg_l1 = sum(l1_list) / n
    avg_mse = sum(mse_list) / n
    avg_psnr = sum(psnr_list) / n

    print(f"Evaluated {n} random samples")
    print(f"Average L1  : {avg_l1:.6f}")
    print(f"Average MSE : {avg_mse:.6f}")
    print(f"Average PSNR : {avg_psnr:.6f}")
    print("Saved inference results to:", OUT_DIR)

    return x_saved, x_rec_saved


def setup_third_party_paths(repo_root):
    import sys

    sys.path.insert(0, str(repo_root / "Mars_Derrick/third_party/latent-diffusion"))
    sys.path.insert(0, str(repo_root / "Mars_Derrick/third_party/taming-transformers"))


if __name__ == "__main__":
    x, x_rec = test_random_samples(n=1000)

making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 64, 64) = 12288 dimensions.
making attention of type 'vanilla' with 512 in_channels
loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
VQLPIPSWithDiscriminator running with hinge loss.
Loaded /scratch/project_2012747/Data_zarr/Testing_VQGAN/LOBSTER-AAPL-2025-12-24_order_images.zarr.zip with shape (1358265, 3, 32, 32)
Evaluated 1000 random samples
Average L1  : 0.003977
Average MSE : 0.000294
Average PSNR : 38.150627
Saved inference results to: /projappl/project_2012747/Mars_Derrick/custom_code/inference


In [56]:
import math


def psnr(gt, rec):
    mse = torch.mean((gt - rec) ** 2).item()
    if mse == 0:
        return float("inf")
    return 20 * math.log10(1.0 / math.sqrt(mse))


Peak_signal_to_noise_ratio = psnr(gt, rec)

print(Peak_signal_to_noise_ratio)

37.74446198625143


### Found that model trained with downsample size 100 works better (which means more data the better), it provided PSNR(Peak Signal-to-Noise Ratio) >38 DB

In [ ]:
import torch
import zarr
import numpy as np
import sys
import os
from pathlib import Path
from zarr.storage import ZipStore
from torch.utils.data import Dataset
from omegaconf import OmegaConf
import bisect
import pytorch_lightning as pl
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split
from pytorch_lightning.callbacks import ModelCheckpoint
from omegaconf import OmegaConf
from zarr.storage import ZipStore
from torchvision.utils import save_image

import re

Best_model = None
# --- FolderZarrDataset (as you already defined) ---


class VQForOrders(pl.LightningModule):
    """VQGAN model class, adapted for Orders"""

    def __init__(self, vqmodel, lr=1e-4):
        super().__init__()
        self.m = vqmodel
        self.lr = lr

    def forward(self, x):
        q, _, info = self.m.encode(x)

        return self.m.decode(q)

    def training_step(self, batch, _):
        x = batch
        x_rec = self(x)
        loss = ((x_rec - x) ** 2).mean()
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x = batch
        x_rec = self(x)
        val_loss = ((x_rec - x) ** 2).mean()
        self.log("val_loss", val_loss, prog_bar=True)
        return val_loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)


class FolderZarrDataset(Dataset):
    def __init__(self, folder_path):
        folder_path = Path(folder_path)
        self.zarr_files = sorted(folder_path.glob("*.zip"))  # <--- keep this
        if not self.zarr_files:
            raise ValueError(f"No .zip files found in {folder_path}")

        self.arrays = []
        self.lengths = []

        for path in self.zarr_files:
            zarr_name = path.name.replace(".zip", "")
            dataset_name = f"{zarr_name}/images"

            store = ZipStore(str(path), mode="r")
            arr = zarr.open(store=store, path=dataset_name, mode="r")

            print(f"Loaded {path} with shape {arr.shape}")

            self.arrays.append(arr)
            self.lengths.append(arr.shape[0])

        # cumulative lengths for indexing
        self.cum_lengths = []
        total = 0
        for l in self.lengths:
            total += l
            self.cum_lengths.append(total)

    def __len__(self):
        return self.cum_lengths[-1]

    def __getitem__(self, idx):
        day_idx = bisect.bisect_right(self.cum_lengths, idx)
        local_idx = idx if day_idx == 0 else idx - self.cum_lengths[day_idx - 1]

        img = self.arrays[day_idx][local_idx]
        img = torch.from_numpy(img).float()
        img = img / 127.5 - 1.0  # normalize [-1,1]

        return img


def load_best_model(best_ckpt_path, repo_root):
    from ldm.util import instantiate_from_config

    cfg = OmegaConf.load(repo_root / "Mars_Derrick/third_party/latent-diffusion/models/first_stage_models/vq-f4/config.yaml")

    vq = instantiate_from_config(cfg.model)

    model = VQForOrders.load_from_checkpoint(checkpoint_path=best_ckpt_path, vqmodel=vq, lr=1e-4, strict=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

    model.to(device)
    model.eval()

    return model, device


def find_best_checkpoint(ckpt_dir: Path) -> Path:
    ckpts = list(ckpt_dir.glob("*.ckpt"))
    if not ckpts:
        raise FileNotFoundError(f"No .ckpt files found in {ckpt_dir}")

    def extract_val_loss(p):
        m = re.search(r"val_loss=([0-9]+(?:\.[0-9]+)?)", p.name)
        if m is None:
            raise ValueError(f"Could not parse val_loss from filename: {p.name}")
        return float(m.group(1))

    best_ckpt = min(ckpts, key=extract_val_loss)
    return best_ckpt


# -------------------------------
# --- Parameters / Setup --------
# -------------------------------
dataset_path = "/projappl/project_2012747/Mars_Derrick/custom_code/training/to_be_converted/"
dataset = FolderZarrDataset(dataset_path)

OUT_DIR = Path("./latents_zip")
OUT_DIR.mkdir(exist_ok=True)

THIS_DIR = Path(os.getcwd()).resolve().parent
REPO = THIS_DIR.parents[1]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sys.path.insert(0, str(REPO / "Mars_Derrick" / "third_party" / "latent-diffusion"))
sys.path.insert(0, str(REPO / "Mars_Derrick" / "third_party" / "taming-transformers"))

# Finding best model
BEST_CKPT_PATH = Path("/scratch") / "project_2012747" / "Mars_Derrick" / "checkpoints" / "checkpoint_downsample_100/"
BEST_CKPT = find_best_checkpoint(BEST_CKPT_PATH)


# ---- Load model ----
model, device = load_best_model(BEST_CKPT, REPO)
Best_model = model

model = model.to(device)
model.eval()

BATCH_SIZE = 1024  # adjust according to GPU memory

# -------------------------------
# --- Process each Zarr file ----
# -------------------------------
for file_idx, arr in enumerate(dataset.arrays):
    num_samples = arr.shape[0]
    print(num_samples)

    print(f"\nProcessing file {file_idx + 1}/{len(dataset.arrays)} with {num_samples} samples")

    # --- preallocate Zarr ZipStore ---
    original_file_name = Path(dataset.zarr_files[file_idx]).stem
    out_file = OUT_DIR / f"{original_file_name}_latents.zarr.zip"

    # --- preallocate Zarr ZipStore for writing ---
    store = ZipStore(str(out_file), mode="w")  # 'w' for write
    latent_arr = zarr.zeros(shape=(num_samples, 3, 8, 8), chunks=(BATCH_SIZE, 3, 8, 8), dtype="f4", store=store, overwrite=True)

    next_report = 100_000

    for start in range(0, num_samples, BATCH_SIZE):
        end = min(start + BATCH_SIZE, num_samples)

        batch_imgs = arr[start:end]  # NumPy view

        x = torch.from_numpy(batch_imgs).to(device=device, dtype=torch.float32)
        x = x.div_(127.5).sub_(1.0)
        x_input = x.add(1).mul_(0.5)

        with torch.inference_mode():
            quant, _, info = model.m.encode(x_input)
            indices = info["indices"]  # or info[2], depends on implementation
            tokens = indices.view(x.size(0), -1)

        latent_arr[start:end] = quant.cpu().numpy()

        if end >= next_report:
            print(f"Processed {end}/{num_samples}")
            next_report += 100_000

    store.close()
    print(f"\nSaved latent Zarr: {out_file} with shape {latent_arr.shape}")

Loaded /projappl/project_2012747/Mars_Derrick/custom_code/training/to_be_converted/LOBSTER-GOOGL-2025-12-16_order_images.zarr.zip with shape (18751982, 3, 32, 32)
Loaded /projappl/project_2012747/Mars_Derrick/custom_code/training/to_be_converted/LOBSTER-TSLA-2025-12-12_order_images.zarr.zip with shape (10442283, 3, 32, 32)
Loaded /projappl/project_2012747/Mars_Derrick/custom_code/training/to_be_converted/LOBSTER-TSLA-2025-12-17_order_images.zarr.zip with shape (10859624, 3, 32, 32)
Loaded /projappl/project_2012747/Mars_Derrick/custom_code/training/to_be_converted/LOBSTER-TSLA-2025-12-22_order_images.zarr.zip with shape (5548388, 3, 32, 32)
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 64, 64) = 12288 dimensions.
making attention of type 'vanilla' with 512 in_channels


/PUHTI_TYKKY_Quvj2Tb/miniforge/envs/env1/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/PUHTI_TYKKY_Quvj2Tb/miniforge/envs/env1/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
VQLPIPSWithDiscriminator running with hinge loss.
18751982

Processing file 1/4 with 18751982 samples
Processed 100352/18751982
Processed 200704/18751982
Processed 300032/18751982
